<a href="https://colab.research.google.com/github/yuliethzapata/Analitica-de-Negocios/blob/main/%C3%81rbol_de_Decisi%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Caso de Estudio**
Una entidad financiera (FLINTECH) quiere implementar un modelo de árbol de decisión para mejorar la preaprobación de créditos de consumo de sus solicitantes de este tipo de créditos. Para este proceso vamos a utilizar las variables:
* Edad: Indica el número de años que posee una persona, o el tiempo que usted lleva en el sistema financiero.
* Ingresos: Engloba todos los ingresos que recibe una persona además si posee salario mensual (UDS).
* Egresos: Continuar describiendo
* Monto (EAD): Continuar describiendo

0. Se procede con la carga de las librerias de trabajo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

1. Se procede con la carga de datos de trabajo

In [ ]:
nxl='/content/1. SolicitantesCrédito(USD).xlsx'
XDB=pd.read_excel(nxl,sheet_name=1)
XD=XDB.iloc[:,[1,10,11,25]]
yd=XDB.iloc[:,32]
display(XD)

,Edad,Ingresos,Egresos,Monto (EAD)
0,38,1356.14400,1685.622667,625.562230
1,51,286.01600,364.462000,140.031984
2,37,517.46325,629.208889,284.564492
3,29,473.27000,551.616889,309.647588
4,42,750.09175,806.715778,500.663578
...,...,...,...,...
5837,48,1207.84800,753.801111,748.041791
5838,31,1472.77200,953.812889,870.793819
5839,38,773.01975,672.910667,594.947894
5840,43,635.50175,780.691556,305.580539


2. Se procede con la implementación del modelo de árbol

In [ ]:
mar=DecisionTreeClassifier(criterion='gini',max_depth=4)
mar.fit(XD,yd) #Aquí el modelo busca la relación entrada-salida

#¿y que fué lo que hizo el modelo?
ydp=mar.predict(XD) #Esto es lo que pronóstica el modelo

#Se construye la matriz de confusón
cm=confusion_matrix(yd,ydp)
display(cm)
VN=cm[0,0]; FP=cm[0,1]; FN=cm[1,0]; VP=cm[1,1]
#Médtricas de Desempeño
Ex=(VP+VN)/len(XD) #1. Exaltitud: Comportamiento General
__builtins__.print("La exaltitud es:",Ex)
sen=VP/(VP+FN) #2. Sensibilidad: Como se comporta pronósticando PreApr
__builtins__.print("La sensibilidad es:",sen)
Spe=VN/(VN+FP) #3. Especifidad: Como se comporta frente a los negativos
__builtins__.print("La especificidad es:", Spe)
pre=VP/(VP+FP) #4. Precisión: Como se comporta pronósticando PreApr
__builtins__.print("La precisión es:",pre)
PreNeg=VN/(VN+FN) #5. Precisión Negativa: Como se comporta pronósticando negativos
__builtins__.print("La predicción negativa es:",PreNeg)

array([[2301,  658],
       [ 644, 2239]])

La exaltitud es: 0.7771311194796303
La sensibilidad es: 0.7766215747485259
La especificidad es: 0.7776275768840825
La precisión es: 0.772868484639282
La predicción negativa es: 0.7813242784380305


3. Despliegue del Árbol de Decisión

In [ ]:
from sklearn.tree import export_graphviz #Exporta los datos a un gráfico
from pydotplus import graph_from_dot_data #Es un graficador

vs=["Edad","Ingresos","Egresos","Monto"] #Títulos del árbol
dot_data=export_graphviz(mar,feature_names=vs) #Exportar de números a gráfico en pdf
graph=graph_from_dot_data(dot_data)               #Hacemos el gráfico
graph.write_png('Arbol.png')

True

**Análisis de Resultados**
De la base de datos se puede observar un total de 2959 solicitantes de crédito que poseen la categoría de PreNeg, mientras que la categoría de PreApr la matriz cuenta con un total de 2883 créditos PreApro. De los 2959 datos el modelo pronóstivo correctamente un total de 2301 datos. de los 2883 PreApr el modelo pronosticó correctamente un total de 2239. El modelo logra identificar en un 77% los créditos PreNeg (2301/2959), mientras que el modelo logró identificar en un 77.66% los créditos PreApr (2239/2883).

Con respecto a las métricas podemos observar que el modelo logró Exactitud, 77,71%, lo que indica el buen comportamiento general del modelo frente a la clasificacion de créditos en las dos categorías de PreApr. Se destacan la especificidad y la precisión las cuales lograron valores en promedio por encima del 77%, lo que supera el limite inferior definido para este tipo de modelo de clasificacion en el cual se ubica en el 75%.

De acuerdo con el árbol de decisión se destaca un nodo puro (10/0) el cual posee la siguiente regla de desición, Si una persona cumple con esa:
SI Monyo<=322 and Ing>376 and Egre<=685 and Ing>644 (923/5)
Tendrá una probabilidad de aprobación del 100%. Es importante destacar que los nodos puros poseen un gini=0 (El modelo diferencia muy bien los buenapagas).

Se destacan en este árbol a pesar de la no existencia de más nodos puros, se destacan los nodos extremos o las reglas del negocio extremas. Se destaca una segunda regla que logra un % de aprobacion del 99%, y la cual posee la siguiente extructura: SI el monto <=322 and Ing<=376 and Monto<=178 and Ing<=232. Por su parte, el nodo derecho posee un % de negación del 98%, donde la regla se define: Monto>322 ans Ing>896 nd Monto>961 and Ing>1178 (7/576)  